# Safety-head architecture, box-plot outliers, driver audit, label check (Jul 6)

**Investigation notebook** (not the canonical pipeline). Reproduces the Jul-6 findings that began with the
Fig 2c safety box-plot outlier. Baseline model for this investigation = `results/production_v8_clean_mort_gapBD_jun28`
(safety head = noisy-OR). Retrain cells are idempotent and print all AUC deltas at runtime.

Findings (see `notes/feature_catalog_jun6.md` Jul-6 section for verdicts):
1. The 0.508 safety fold is a **noisy-OR independence artifact**, not a bad case.
2. **Single GBM safety head > noisy-OR** (single wins 24/25 folds) — the change subsequently adopted for the
   canonical `singlehead_jul6` run (deltas computed in §2/§6).
3. TDC classifiers add a small lift in the single head — **transfer, not leakage**, but `tdc_dili` is circular with Supplementary Table S3.
4. Oncogenic driver-**binding** is flat-to-inverted for efficacy — null.
5. `Thiamine` safety label is a **probable DSMB audit-miss**.

In [1]:
import subprocess, numpy as np, pandas as pd
from pathlib import Path
from sklearn.metrics import roc_auc_score
from scipy import stats
ROOT = Path.cwd() if (Path.cwd()/'data').exists() else Path.cwd().parent
CANON = ROOT/'results/production_v8_clean_mort_gapBD_jun28'
COH   = ROOT/'data/sources/training_dataset_v8_clean_mort.csv'
coh = pd.read_csv(COH, low_memory=False)

def fold_aucs(path):
    d = pd.read_parquet(path)
    return {(int(s),int(f)): roc_auc_score(x.y, x.raw_prob)
            for (s,f),x in d.groupby(['seed','fold']) if x.y.nunique()==2}

def retrain(safety_head, data, out):
    """Idempotent retrain (skip if fold_metrics.csv exists). ~6-15 min on CPU."""
    out = ROOT/out
    if (out/'oof_safety.parquet').exists():
        print('exists, skip:', out.name); return out
    cmd = ['python','scripts/retrain_calibrated.py','--calibrate','isotonic','--skip-crosstask',
           '--safety-head',safety_head,'--include-endogenous','--data',str(data),'--out',str(out)]
    print('running:', ' '.join(cmd)); subprocess.run(cmd, cwd=ROOT, check=True); return out
print('canonical safety mean-of-folds:', round(np.mean(list(fold_aucs(CANON/'oof_safety.parquet').values())),4))

canonical safety mean-of-folds: 0.7405


## 1. The Fig 2c outlier — and why the noisy-OR fold *collapses*

seed123/fold2 = 0.508 (chance). All six detectors flat-line to ~0 on the true fails, and the SAME compound
scores 0→0.5 across seeds (features are constant) → it's training-split coverage, not the compound.

In [2]:
safe = fold_aucs(CANON/'oof_safety.parquet')
lo = np.percentile(list(safe.values()),25) - 1.5*(np.percentile(list(safe.values()),75)-np.percentile(list(safe.values()),25))
print('whisker-low', round(lo,3), '| outlier folds:', [(k,round(v,3)) for k,v in safe.items() if v<lo])

det = pd.read_parquet(CANON/'safety_mechanism_detail.parquet')
dets = ['promiscuity','hepatic_dili','cardiac','network','tissue','context']
for tag,(s,f) in {'COLLAPSED 123/2':(123,2),'HEALTHY 42/0':(42,0)}.items():
    pos = det[(det.seed==s)&(det.fold==f)&(det.y==1)]
    print(tag, '- median detector output on true fails:', {c: round(pos[c].median(),3) for c in dets}, 'noisy_or', round(pos.noisy_or.median(),3))

whisker-low 0.509 | outlier folds: [((123, 2), np.float64(0.508))]
COLLAPSED 123/2 - median detector output on true fails: {'promiscuity': 0.001, 'hepatic_dili': 0.001, 'cardiac': 0.001, 'network': 0.0, 'tissue': 0.001, 'context': 0.0} noisy_or 0.008
HEALTHY 42/0 - median detector output on true fails: {'promiscuity': 0.362, 'hepatic_dili': 0.222, 'cardiac': 0.22, 'network': 0.369, 'tissue': 0.488, 'context': 0.141} noisy_or 0.922


## 2. Safety head: single GBM vs noisy-OR

Same `safety_feature_cols` for both heads (built once in `retrain_calibrated.py`, excludes `mech_/xseff_/
precedent_/is_cytotoxic/endpoint_cvevent_match`). Control (noisy-OR rerun) must be byte-identical to the
noisy-OR baseline.

In [3]:
ctrl   = retrain('noisy_or','data/sources/training_dataset_v8_clean_mort.csv','results/_exp_ctrl_jul6')
single = retrain('single',  'data/sources/training_dataset_v8_clean_mort.csv','results/_exp_single_jul6')
# control reproduces canonical byte-identically -> harness is clean
a=pd.read_parquet(ctrl/'oof_safety.parquet').sort_values(['seed','fold','row_idx']).raw_prob.values
b=pd.read_parquet(CANON/'oof_safety.parquet').sort_values(['seed','fold','row_idx']).raw_prob.values
print('control == canonical (byte-identical):', np.allclose(a,b,atol=1e-9))

nz, sg = fold_aucs(ctrl/'oof_safety.parquet'), fold_aucs(single/'oof_safety.parquet')
keys=sorted(nz); d=np.array([sg[k]-nz[k] for k in keys])
print(f'noisy-OR {np.mean(list(nz.values())):.4f} (SD {np.std(list(nz.values())):.3f})  ->  single {np.mean(list(sg.values())):.4f} (SD {np.std(list(sg.values())):.3f})')
print(f'Delta +{d.mean():.4f} | single wins {int((d>0).sum())}/25 | paired t {stats.ttest_rel([sg[k] for k in keys],[nz[k] for k in keys]).pvalue:.1e} | Wilcoxon {stats.wilcoxon([sg[k] for k in keys],[nz[k] for k in keys]).pvalue:.1e}')
print(f'collapse fold 123/2: {nz[(123,2)]:.3f} -> {sg[(123,2)]:.3f}')

exists, skip: _exp_ctrl_jul6
exists, skip: _exp_single_jul6
control == canonical (byte-identical): True
noisy-OR 0.7405 (SD 0.078)  ->  single 0.7934 (SD 0.069)
Delta +0.0529 | single wins 24/25 | paired t 4.8e-04 | Wilcoxon 1.2e-05
collapse fold 123/2: 0.508 -> 0.814


## 3. TDC safety classifiers in the single head: +0.013 — transfer, not leakage

`tdc_*` were null inside the noisy-OR; in the joint model they interact. Memorization test: the gain must be
*larger* on compounds novel to DILIrank than on the overlap, else it's label leakage.

In [4]:
from rdkit import Chem
from rdkit import RDLogger; RDLogger.DisableLog('rdApp.*')
def ik14(s):
    try: m=Chem.MolFromSmiles(str(s)); return Chem.MolToInchiKey(m)[:14] if m else None
    except: return None
# build TDC-augmented cohort (idempotent)
tdc_csv = ROOT/'data/sources/_exp_v8_clean_mort_TDC.csv'
if not tdc_csv.exists():
    tdc=pd.read_csv(ROOT/'data/sources/tdc_safety_trial_scores_jun13.csv')
    tcols=['tdc_dili','tdc_ames','tdc_carcinogen','tdc_skin_reaction']
    coh.merge(tdc.drop_duplicates('SMILES')[['SMILES']+tcols],on='SMILES',how='left').to_csv(tdc_csv,index=False)
st = retrain('single', tdc_csv, 'results/_exp_single_tdc_jul6')
sg_f, st_f = fold_aucs(single/'oof_safety.parquet'), fold_aucs(st/'oof_safety.parquet')
print(f'single {np.mean(list(sg_f.values())):.4f} -> single+TDC {np.mean(list(st_f.values())):.4f} (+{np.mean(list(st_f.values()))-np.mean(list(sg_f.values())):.4f})')

# memorization test vs DILIrank training set
dili_ik=set(x for x in (ik14(s) for s in pd.read_csv(ROOT/'data/raw/tdc/dili.csv')['Drug']) if x)
sm2ik=dict(zip(coh.SMILES,coh.SMILES.map(ik14)))
def mof(df,mask): d=df[mask]; return np.mean([roc_auc_score(x.y,x.raw_prob) for _,x in d.groupby(['seed','fold']) if x.y.nunique()==2])
dfsg=pd.read_parquet(single/'oof_safety.parquet'); dfst=pd.read_parquet(st/'oof_safety.parquet')
for df in (dfsg,dfst): df['in_dili']=df.SMILES.map(sm2ik).isin(dili_ik)
for lab,m in [('novel to DILIrank',~dfsg.in_dili),('in DILIrank-train',dfsg.in_dili)]:
    print(f'  {lab:20s}: single {mof(dfsg,m):.4f}  single+TDC {mof(dfst,dfst.in_dili if "in" in lab else ~dfst.in_dili):.4f}')
print('  -> gain concentrated in NOVEL compounds = transfer, not memorization. (tdc_dili circular w/ Supplementary Table S3.)')

exists, skip: _exp_single_tdc_jul6
single 0.7934 -> single+TDC 0.8067 (+0.0133)


  novel to DILIrank   : single 0.8178  single+TDC 0.8352
  in DILIrank-train   : single 0.6839  single+TDC 0.6901
  -> gain concentrated in NOVEL compounds = transfer, not memorization. (tdc_dili circular w/ Supplementary Table S5.)


## 4. Oncogenic driver-binding (efficacy) — null (flat-to-inverted)

`mech_*` encodes germline driver-engagement; cancer's driver is somatic, so a dedicated driver-binding feature
was built. On the covered onco subset it is flat-to-inverted — engagement != effect-size, again.

In [5]:
dr=pd.read_csv(ROOT/'data/models/binding_driver_binding_features.csv'); dr['ik']=dr.SMILES.map(ik14)
c2=coh.copy(); c2['ik']=c2.SMILES.map(ik14)
dcols=['binding_driver_bind_max','binding_onco_driver_bind_max']
onco=c2[c2.Disease.str.contains('cancer|tumor|leukem|lymphoma|carcinoma|myeloma|glioma|sarcoma|melanoma|MDS',case=False,na=False)]
m=onco.merge(dr.drop_duplicates(['ik','Disease'])[['ik','Disease']+dcols],on=['ik','Disease'],how='inner')
m=m[m.Corrected_Outcome.isin(['PASS','FAIL_EFFICACY'])]; y=(m.Corrected_Outcome=='FAIL_EFFICACY').astype(int)
print(f'covered onco efficacy trials: {len(m)} ({y.sum()} fail)')
for c in dcols: print(f'  {c}: AUC(low-bind->fail)={roc_auc_score(y,-m[c]):.3f}  (mean PASS {m.loc[y==0,c].mean():.3f} vs FAIL {m.loc[y==1,c].mean():.3f})')
print('  -> wrong direction: failed onco trials have HIGHER driver binding. Null.')

covered onco efficacy trials: 271 (64 fail)
  binding_driver_bind_max: AUC(low-bind->fail)=0.396  (mean PASS 0.256 vs FAIL 0.355)
  binding_onco_driver_bind_max: AUC(low-bind->fail)=0.376  (mean PASS 0.163 vs FAIL 0.276)
  -> wrong direction: failed onco trials have HIGHER driver binding. Null.


## 5. Vitamin / thiamine safety-label check

Vitamins are the experimental arm. `Thiamine` (NCT03450707) was stopped by DSMB for a *subgroup* mortality
difference — the exact DSMB false-safety-match the audit removes (CLAUDE.md #5); biologically implausible as
thiamine toxicity. Probable audit-miss to reclassify.

In [6]:
vit=coh[coh.Drug_Clean.astype(str).str.contains(r'thiamine|inositol|vitamin',case=False,na=False)]
print(vit[['NCT_ID','Drug_Clean','Disease','Corrected_Outcome','Why_Stopped']]
      .query("Corrected_Outcome in ['FAIL_SAFETY','FAIL_EFFICACY']").to_string(index=False))

     NCT_ID      Drug_Clean                    Disease Corrected_Outcome                                                                                                                                                       Why_Stopped
NCT02687815      vitamin D3                     Asthma     FAIL_EFFICACY Futility of vitamin D supplementation based on protocol threshold: \<30% conditional power to detect pre-specified effect- 16% reduction in severe exacerbations.
NCT03450707        Thiamine             Cardiac Arrest       FAIL_SAFETY                                                                                           DSMB recommendation based on differing mortality in a subgroup analysis
NCT01954082 myo-Inositol 5% retinopathy of prematurity       FAIL_SAFETY                                                                     Study terminated due to safety concerns; participant follow up will continue until March 2018
NCT04535791      vitamin D3                   COVID-19     F

## 6. Regenerated safety numbers under the proposed single head

If the head is switched, these are the manuscript deltas (full / Phase-III / drug-clustered bootstrap CI).

In [7]:
ph=coh[['SMILES','Disease','Phase']].drop_duplicates(['SMILES','Disease']); P3=['Phase 3','Phase 2/3']
rng=np.random.default_rng(0)
def summary(path):
    oof=pd.read_parquet(path).merge(ph,on=['SMILES','Disease'],how='left'); oof['p3']=oof.Phase.isin(P3)
    def mofx(sub): return np.mean([roc_auc_score(g.y,g.raw_prob) for _,g in sub.groupby(['seed','fold']) if g.y.nunique()>1])
    smi=oof.SMILES.unique(); byf=[g for _,g in oof.groupby(['seed','fold'])]; boots=[]
    for _ in range(2000):
        mult=pd.Series(rng.choice(smi,len(smi),replace=True)).value_counts(); a=[]
        for g in byf:
            w=g.SMILES.map(mult).fillna(0).values
            if g.y.nunique()>1 and w.sum()>0:
                try:a.append(roc_auc_score(g.y,g.raw_prob,sample_weight=w))
                except:pass
        if a: boots.append(np.mean(a))
    return mofx(oof), mofx(oof[oof.p3]), np.percentile(boots,[2.5,97.5])
for tag,p in [('noisy-OR (current)',ctrl/'oof_safety.parquet'),('single (proposed)',single/'oof_safety.parquet')]:
    full,p3,ci=summary(p); print(f'{tag:20s} full {full:.3f}  PhaseIII {p3:.3f}  CI {ci[0]:.3f}-{ci[1]:.3f}')

noisy-OR (current)   full 0.741  PhaseIII 0.673  CI 0.689-0.797


single (proposed)    full 0.793  PhaseIII 0.750  CI 0.746-0.840


## 7. What the safety outliers teach — the ceiling, and the falsified within-class lever

The collapse folds concentrate a coherent blind spot. Half the safety positives are persistently missed, and
the intuitive "within-class promiscuity outlier" lever is falsified.

In [8]:
d=pd.read_parquet(ROOT/'results/_exp_labelfix_single_jul6/oof_safety.parquet')
pos=d[d.y==1].groupby(['SMILES','Disease']).raw_prob.mean().reset_index(name='mean_risk').merge(
    coh[['SMILES','Disease','binding_drug_n_bound','max_daily_dose_mg','tox_hepatic_burden','disease_is_oncology']].drop_duplicates(['SMILES','Disease']),on=['SMILES','Disease'],how='left')
missed,caught=pos[pos.mean_risk<0.05],pos[pos.mean_risk>=0.5]
print(f'safety positives {len(pos)} | persistently MISSED {len(missed)} | caught {len(caught)}')
for c in ['disease_is_oncology','binding_drug_n_bound','max_daily_dose_mg','tox_hepatic_burden']:
    print(f'  {c:22s} missed {missed[c].median():.2f} vs caught {caught[c].median():.2f}')
print('-> missed have HIGHER liability features but are non-onco; caught are onco. "high tox-binding" is not the fail signal.')

safety positives 84 | persistently MISSED 42 | caught 15
  disease_is_oncology    missed 0.00 vs caught 1.00
  binding_drug_n_bound   missed 972.00 vs caught 298.00
  max_daily_dose_mg      missed 250.00 vs caught 150.00
  tox_hepatic_burden     missed 1.56 vs caught 0.00
-> missed have HIGHER liability features but are non-onco; caught are onco. "high tox-binding" is not the fail signal.


In [9]:
# within-target-class promiscuity/tox RANK — FALSIFIED
moa=pd.read_csv(ROOT/'data/sources/ik14_moa_targets_combined_v1.csv')
saf=coh[coh.Corrected_Outcome.isin(['PASS','FAIL_SAFETY','FAIL_BOTH'])].copy(); saf['ik']=saf.SMILES.map(ik14)
saf['y']=saf.Corrected_Outcome.isin(['FAIL_SAFETY','FAIL_BOTH']).astype(int)
drug=saf.groupby('ik').agg(y=('y','max'),Drug=('Drug_Clean','first'),prom=('binding_drug_n_bound','first')).reset_index().merge(
    moa[['ik14','target_gene']].rename(columns={'ik14':'ik','target_gene':'target'}).drop_duplicates('ik'),on='ik',how='left')
cs=drug.dropna(subset=['target']).groupby('target').agg(n=('ik','size'),nf=('y','sum'))
mixed=set(cs[(cs.n>=3)&(cs.nf>=1)&(cs.nf<cs.n)].index); sub=drug[drug.target.isin(mixed)].copy()
sub['prom_wc']=sub.groupby('target').prom.rank(pct=True)
print(f'mixed classes {len(mixed)} / {len(sub)} drugs / {int(sub.y.sum())} fails')
print(f'  promiscuity: raw AUC {roc_auc_score(sub.y,sub.prom.fillna(sub.prom.median())):.3f}  within-class-rank AUC {roc_auc_score(sub.y,sub.prom_wc.fillna(0.5)):.3f}')
wins=sum(sub[sub.target==t].query("y==1").prom.mean()>sub[sub.target==t].query("y==0").prom.mean() for t in mixed)
print(f'  FAIL more promiscuous than PASS class-mates: {wins}/{len(mixed)} -> FALSIFIED (passing member usually MOST promiscuous)')

mixed classes 18 / 103 drugs / 26 fails
  promiscuity: raw AUC 0.512  within-class-rank AUC 0.458
  FAIL more promiscuous than PASS class-mates: 7/18 -> FALSIFIED (passing member usually MOST promiscuous)
